[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/48_policy_model_solution.ipynb)

#  Solution: Policy & Value Head Configuration

Reference: PPO (Schulman et al.), GRPO, and standard actor-critic architectures.

```text
Policy head: Linear(h, hidden) -> ReLU -> Linear(hidden, action_dim)
Value head:  Linear(h, hidden) -> ReLU -> Linear(hidden, 1)
Both use Kaiming uniform init (fan_in mode, gain=sqrt(2) for ReLU).
```

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn

In [ ]:
#  SOLUTION

def configure_policy_model(base_model, action_dim, hidden_dim=256):
    """Add policy and value heads to a base model.

    Following the standard actor-critic pattern used in PPO/GRPO:
    - Policy head: output logits over actions (for discrete) or action params (continuous)
    - Value head: output scalar state value estimate
    - Kaiming init ensures proper gradient flow through ReLU
    Reference: Schulman et al. 2017; DeepSeek-R1 GRPO training setup.
    """
    d_model = base_model.linear.out_features

    policy_head = nn.Sequential(
        nn.Linear(d_model, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, action_dim),
    )
    value_head = nn.Sequential(
        nn.Linear(d_model, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, 1),
    )

    # Kaiming He initialization (standard for ReLU activations)
    for head in [policy_head, value_head]:
        for m in head.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    return policy_head, value_head

In [ ]:
#  Verify
class DummyBase(nn.Module):
    def __init__(self): super().__init__(); self.linear = nn.Linear(32, 64)
p, v = configure_policy_model(DummyBase(), 4, 32)
x = torch.randn(2, 5, 32)
h = DummyBase()(x)
print(f"Policy logits: {p(h).shape}, Values: {v(h).shape}")

In [ ]:
from torch_judge import check
check("policy_model")